In [1]:
from utils.modeling import Qwen3VLSparseForConditionalGeneration


/workspace/conda/envs/vlm_node_1016/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from transformers import AutoConfig,AutoProcessor

MODEL_ID = "Aasdfip/qwen3_webnav_0.1" #,"Qwen/Qwen3-VL-2B-Instruct" # Or your specific VLM backbone

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen3VLSparseForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    config=config,
    # device='cuda',
    device_map={"": 1},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation = "flash_attention_2",
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
from utils.collators import ActionMaskingVLMCollator
from utils.data_misc import decode_image_sequence
from datasets import load_dataset
ds = load_dataset("tccoin/navverse-benchmark",streaming=True)['train']
ds = ds.shuffle(seed=42, buffer_size=1000)

ds = ds.map(decode_image_sequence)
data_collator=ActionMaskingVLMCollator(
    processor=processor,
    length_warning = 100000,
    # max_length=TOTAL_BUDGET,
    dropout=0.3,
)

In [11]:
ds_iter = iter(ds)

In [15]:
inputs = data_collator([next(ds_iter)])
inputs.pop('labels')

DEBUG PRE-COLLATE: [(32, (640, 480))]
input id shape: torch.Size([1, 10467])


tensor([[  -100,   -100,   -100,  ...,    334, 151645,   -100]])

tensor([[  -100,   -100,   -100,  ...,    334, 151645,   -100]])

In [16]:
model(**{k:v.to('cuda:1') for k,v in inputs.items()},logits_to_keep=1)

OutOfMemoryError: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 1 has a total capacity of 79.14 GiB of which 108.75 MiB is free. Process 3678384 has 79.02 GiB memory in use. Of the allocated memory 77.69 GiB is allocated by PyTorch, and 853.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

torch.Size([1, 13019])